# LiNeMesh: training

This notebook trains the LiNeMesh mesh-generation network on BridgeNet (or on your own synthetic domains).

**Before running**
1. Install the requirements: `pip install -r requirements.txt`
2. Download the training data: `python scripts/download_bridgenet.py --split train`
3. Run all cells. Paths come from `LM_paths.py`, so nothing needs to be edited.

* Training data: `data/bridgenet/train/<id>/<id>_render/` (to train on your synthetic data instead, set `DATA_DIR = SYNTH_DIR / "train"` in the data-loading cell).
* The best checkpoint is saved to `weights/mod_lm_brdnet.pth`.
* A CUDA GPU is strongly recommended. The ImageNet weights of ResNet-34/101 are downloaded by torchvision on first use.

In [ ]:
## ---- repository paths (works from the repository root or any subfolder) ----
import os, sys
from pathlib import Path
_root = Path.cwd()
while not (_root / "LM_paths.py").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))
from LM_paths import TRAIN_DIR, SYNTH_DIR, WEIGHTS_DIR, TRAINED_WEIGHTS, list_domains
print("repository:", _root)

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import cv2
import random
import torch.optim as optim
import matplotlib.pyplot as plt
from IPython.display import clear_output
import math
from scipy.spatial.transform import Rotation as R
from torch.nn.functional import grid_sample
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import resnet34, ResNet34_Weights
from torchvision.models import resnet101, ResNet101_Weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.ops import FeaturePyramidNetwork


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
roll_c_u, pitch_c_u, yaw_c_u = -(3.141592/2), 0, 3.141592/2 
rx = np.matrix([[1, 0, 0],[0, math.cos(roll_c_u),-math.sin(roll_c_u)], [0, math.sin(roll_c_u), math.cos(roll_c_u)]])
ry = np.matrix([[math.cos(pitch_c_u), 0, math.sin(pitch_c_u)],[0, 1, 0],[-math.sin(pitch_c_u), 0, math.cos(pitch_c_u)]])
rz = np.matrix([[math.cos(yaw_c_u), -math.sin(yaw_c_u), 0],[math.sin(yaw_c_u), math.cos(yaw_c_u), 0],[0, 0, 1]])
R_c_b = np.matmul(np.matmul(rz,ry),rx)

In [ ]:
import torchvision.transforms as T

In [ ]:
def agg_feas(x, mask=None, use_variance=False):
    """
    Aggregates features across dimension 0.

    Args:
        x : Tensor
            Shape: (S, B, D)

        mask : Bool tensor or None
            Shape: (B, S)
            True  -> invalid
            False -> valid

        use_variance : bool
            False -> mean aggregation
            True  -> variance aggregation

    Returns:
        Tensor of shape (B, D)
    """

    eps = 1e-6
    if mask is not None:
        valid = (~mask).float().T.unsqueeze(2)
        count = valid.sum(0).clamp(min=eps)
        mean = (x * valid).sum(0) / count
        if use_variance:
            diff2 = ((x - mean.unsqueeze(0)) ** 2) * valid
            var = diff2.sum(0) / count
            return var
        return mean
    else:
        mean = x.mean(0)
        if use_variance:
            var = ((x - mean.unsqueeze(0)) ** 2).mean(0)
            return var
        return mean

In [ ]:
class context(nn.Module):
    def __init__(self, embed_dim=512, out_dim=24):
        super().__init__()
        self.fc = nn.Linear(embed_dim, out_dim)
    def forward(self, x_, mask=None):
        x = agg_feas(x_, mask, use_variance=False)
        x = self.fc(x)
        return x

In [ ]:
def normalize_uv(uv, H=480, W=640, sz=300, odim=True):
    if odim:
        u = W*uv[..., 0]
        v = H*uv[..., 1]
        u = (sz/W)*u
        v = (sz/H)*v
    else:
        u = uv[..., 0]
        v = uv[..., 1]

    u_norm = 2.0 * u / (sz - 1) - 1.0
    v_norm = 2.0 * v / (sz - 1) - 1.0
    return torch.stack([u_norm, v_norm], dim=-1)

In [ ]:
class AResNet50Encoder(nn.Module):
    def __init__(self, out_h, out_w, pretrained=True, big=False):
        super().__init__()
        self.out_h = out_h
        self.out_w = out_w
        if big:
            backbone = resnet101(weights=ResNet101_Weights.IMAGENET1K_V2)
            self.proj = nn.Conv2d(2048, 3, kernel_size=1)
        else:
            backbone = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
            self.proj = nn.Conv2d(512, 3, kernel_size=1)
        
        self.backbone = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )
    def forward(self, x):
        feat = self.backbone(x)
        feat = self.proj(feat)
        feat = F.interpolate(
            feat,
            size=(self.out_h, self.out_w),
            mode="bilinear",
            align_corners=False,
        )
        return feat

In [ ]:
class SequenceSelfAttention(nn.Module):
    def __init__(
        self,
        channels=3, # 3, 18, 3
        embed_dim=64,
        num_heads=1,
        dropout=0.1
    ):
        super().__init__()
        self.embed = nn.Linear(channels, embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.proj = nn.Linear(embed_dim, channels)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )

    def forward(self, x):
        S, C, H, W = x.shape
        x = x.permute(2, 3, 0, 1)
        x = x.reshape(H * W, S, C)
        x = self.embed(x)
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn = self.ffn(x)
        x = self.norm2(x + ffn)
        x = self.proj(x)
        x = x.reshape(H, W, S, C)
        x = x.permute(2, 3, 0, 1)
        return x

In [ ]:
class ResNet50FPN2(nn.Module):

    def __init__(
        self,
        out_channels=256,
        pretrained=True,
        resv=False,
        in_channels=6
    ):
        super().__init__()
        if resv:
            if pretrained:
                backbone = resnet34(
                    weights=ResNet34_Weights.IMAGENET1K_V1
                )
            else:
                backbone = resnet34(weights=None)

            backbone_channels = [64, 128, 256, 512]

        else:
            if pretrained:
                backbone = resnet101(
                    weights=ResNet101_Weights.IMAGENET1K_V2
                )
            else:
                backbone = resnet101(weights=None)

            backbone_channels = [256, 512, 1024, 2048]
        if in_channels != 3:
            old_conv = backbone.conv1
            new_conv = nn.Conv2d(
                in_channels=in_channels,
                out_channels=old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False
            )
            backbone.conv1 = new_conv

        self.backbone = create_feature_extractor(
            backbone,
            return_nodes={
                "layer1": "c2",
                "layer2": "c3",
                "layer3": "c4",
                "layer4": "c5",
            },
        )

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=backbone_channels,
            out_channels=out_channels,
        )

    def forward(self, x):
        features = self.backbone(x)
        fpn_features = self.fpn(features)
        return fpn_features

In [ ]:
class CorrelationSamplerbatch(nn.Module):
    def __init__(self, in_dim=4900, out_dim=512, w=300, h=300):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(in_dim, 2048, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(2048, out_dim, kernel_size=1)
        )

        self.w = w
        self.h = h

    def forward(self, corr, uv):
        """
        Args:
            corr: M x C_in x H x W
                  e.g. M x 4900 x 70 x 70

            uv:   M x T x 2
                  UV coordinates in the original 300x300
                  image coordinate system.

        Returns:
            feat: M x T x out_dim
        """

        # ---------------------------------------------------------
        # 1. Encode correlation volume
        # ---------------------------------------------------------
        #
        # M x 4900 x 70 x 70
        #       ↓
        # M x out_dim x 70 x 70
        #
        corr = self.encoder(corr)

        M, C, H, W = corr.shape
        T = uv.shape[1]

        uv = uv.float()

        grid = uv.clone()

        grid[..., 0] = (
            2.0 * grid[..., 0] / (self.w - 1)
        ) - 1.0

        grid[..., 1] = (
            2.0 * grid[..., 1] / (self.h - 1)
        ) - 1.0

        # ---------------------------------------------------------
        # 3. Prepare grid for grid_sample
        # ---------------------------------------------------------
        #
        # grid_sample expects:
        #
        # input: M x C x H x W
        # grid:  M x H_out x W_out x 2
        #
        # We want T sampled points, so:
        #
        # M x T x 1 x 2
        #
        grid = grid.unsqueeze(2)

        # ---------------------------------------------------------
        # 4. Batched bilinear interpolation
        # ---------------------------------------------------------
        #
        # output:
        # M x C x T x 1
        #
        feat = F.grid_sample(
            corr,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=True
        )

        # ---------------------------------------------------------
        # 5. Rearrange
        # ---------------------------------------------------------
        #
        # M x C x T x 1
        #     ↓
        # M x T x C
        #
        feat = feat.squeeze(-1).transpose(1, 2)

        return feat

In [ ]:

class ExtendedResidualBlock2(nn.Module):
    def __init__(self, in_features, out_features, n_layers=5):
        super(ExtendedResidualBlock2, self).__init__()
        self.n_layers = n_layers
        self.fc1 = nn.Linear(in_features, out_features)
        self.bn1 = nn.BatchNorm1d(out_features)
        self.shortcut = nn.Sequential()
        if in_features != out_features:
            self.shortcut = nn.Sequential(
                nn.Linear(in_features, out_features))
        self.fcn_dict = nn.ModuleDict()
        self.fcn_dict2 = nn.ModuleDict()
        for i in range(n_layers):
            self.fcn_dict[str(i)] = nn.Sequential(nn.Linear(out_features, out_features))
         
    def forward(self, x):
        out = F.tanh(self.fc1(x)) #torch.sin(self.fc1(x)) #
        for i in range(self.n_layers):
            out = self.fcn_dict[str(i)](out)
            if (i+1)%3==0:
                out = out + self.shortcut(x) #F.tanh(out)
        out = out + self.shortcut(x)
        return out

class EfficientNetwork2(nn.Module):
    def __init__(self, inp, out, n_layers=5):
        super(EfficientNetwork2, self).__init__()
        self.input_layer = nn.Linear(inp, 1024)
        self.residual_block1 = ExtendedResidualBlock2(1024, 512, n_layers)
        self.residual_block2 = ExtendedResidualBlock2(512, 256, n_layers)
        self.residual_block3 = ExtendedResidualBlock2(256, 128, n_layers)
        self.residual_block4 = ExtendedResidualBlock2(128, 64, n_layers)
        self.residual_block5 = ExtendedResidualBlock2(64, 32, n_layers)
        self.fc3 = nn.Linear(32, 16)
        self.output_layer = nn.Linear(16, out)

    def forward(self, x):
        out = self.input_layer(x)
        out = self.residual_block1(out)
        out = self.residual_block2(out)
        out = self.residual_block3(out)
        out = self.residual_block4(out)
        out = self.residual_block5(out)
        out = self.fc3(out)
        out = self.output_layer(out)
        return out


In [ ]:
class base_network(nn.Module):
    def __init__(self,resv=False):
        super().__init__()

        self.out_ch = 256
        self.out_h = 70
        self.out_w = 70
        self.resv = resv
        self.model3 = EfficientNetwork2(1536, 10, 2)

        self.model_fpn = ResNet50FPN2(
            out_channels=self.out_ch,
            resv=resv,
            in_channels=3
        )

        self.cor_model = CorrelationSamplerbatch(
            in_dim=self.out_h * self.out_h,
            out_dim=512
        )

        self.model2 = context(
            embed_dim=1536,
            out_dim=1536)

    def forward(
        self,
        fimg_enc, #imgs_f,#
        imgts,
        uvts,
        viz_mat=None,
        xyzs=None,
        rshp=False,
        UV_feas=None
    ):

        M = imgts.shape[0]
        imgts = imgts.float().to(device, non_blocking=True)
        imgts_70 = F.interpolate(
            imgts,
            size=(self.out_h, self.out_w),
            mode="bilinear",
            align_corners=False
        )

        imgts_flat = imgts_70.flatten(2)
        corr = torch.matmul(
            imgts_flat.transpose(1, 2),   # M x 4900 x 3
            fimg_enc.expand(M, -1, -1)    # M x 3 x 4900
        )
        corr = corr.reshape(M,self.out_h,self.out_w,self.out_h,self.out_w)
        corr = corr.permute(0, 3, 4, 1, 2)
        corr = corr.reshape(M,self.out_h * self.out_w,self.out_h,self.out_w)

        uv = torch.as_tensor(np.stack(uvts),dtype=torch.float32,device=device)
        T = uv.shape[1]
        corr_feas = self.cor_model(corr,uv)

        uv_i = normalize_uv(uv,sz=300,odim=False)
        grid = uv_i.unsqueeze(2)
        if UV_feas is None:
            pyramid = self.model_fpn(imgts)
            multi_scale_feats = []
            for lvl in pyramid.values():
                sampled = F.grid_sample(lvl,grid,mode="bilinear",align_corners=True)
                sampled = sampled.squeeze(-1)
                sampled = sampled.transpose(1, 2)
                multi_scale_feats.append(sampled)
            UV_feas = torch.cat(multi_scale_feats,dim=-1)

        comb_feas = torch.cat([UV_feas, corr_feas],dim=-1)
        if rshp:
            Nb = T // 9
            comb_feas = comb_feas[:, :Nb * 9]
            comb_feas = comb_feas.reshape(M,Nb,9,-1).mean(dim=2)

        if viz_mat is not None:
            mask = (viz_mat == 0).to(device)
        else:
            mask = None

        fess2 = self.model2(comb_feas,mask)
        pred_cls = self.model3(fess2)

        return pred_cls

In [ ]:
class group_occup_sdf(nn.Module):
    def __init__(self):
        super(group_occup_sdf, self).__init__()
        self.out_h, self.out_w = 70, 70
        self.enc_model = AResNet50Encoder(out_h=self.out_h, out_w=self.out_w, big=False)
        self.attnm = SequenceSelfAttention() 
        self.basenet = base_network()
    def forward(self, all_inps_, viz_mat_=None):
        preds_cls = []
        for kk1 in range(len(all_inps_)):
            imgts_, uvts = all_inps_[kk1]
            imgts, imgs_f = imgts_
            fimg_enc = self.attnm(self.enc_model(imgs_f.float().to(device))).mean(0).unsqueeze(0)
            fimg_enc = fimg_enc.reshape(1, 3, self.out_h * self.out_w)
            pred_cls_i = self.basenet(fimg_enc,imgts,uvts,viz_mat_[kk1],rshp=True)
            preds_cls.append(pred_cls_i) 
        preds_cls_ = torch.vstack(preds_cls)
        return preds_cls_

In [ ]:
criterion1 = nn.SmoothL1Loss().to(device)
criterion2 = nn.BCEWithLogitsLoss().to(device)
criterion3 = nn.CrossEntropyLoss().to(device)

In [ ]:
def add_specular_highlight(img, intensity=1.5, radius=20):
    """Adds a bright circular spot to simulate reflection on glass/shiny surfaces."""
    h, w, _ = img.shape
    cx, cy = random.randint(0, w-1), random.randint(0, h-1)
    mask = np.zeros((h, w), dtype=np.float32)
    cv2.circle(mask, (cx, cy), radius, 1, -1)
    mask = cv2.GaussianBlur(mask, (radius//2*2+1, radius//2*2+1), 0)
    highlight = (img * (1 + intensity * mask[..., None])).clip(0, 1)
    return highlight

def add_lens_flare(img, num_streaks=3):
    """Adds simple lens flare streaks."""
    h, w, _ = img.shape
    flare = img.copy()
    cx, cy = random.randint(0, w-1), random.randint(0, h-1)
    for _ in range(num_streaks):
        dx, dy = random.choice([(-1,0),(1,0),(0,-1),(0,1)])
        length = random.randint(50, min(h,w)//2)
        overlay = np.zeros_like(img)
        for i in range(length):
            x, y = cx + dx*i, cy + dy*i
            if 0 <= x < w and 0 <= y < h:
                overlay[y, x] = [1, 1, 1]  # white flare
        flare = np.clip(flare + cv2.GaussianBlur(overlay, (15,15), 0)*0.3, 0, 1)
    return flare

def add_tint(img, color=(0.8, 1.0, 1.2)):
    """Applies a global tint (RGB multiplier)."""
    tinted = (img * np.array(color)[None, None, :]).clip(0, 1)
    return tinted

def pt_trans(img, ph_trans=False):
    """
    img: numpy array of shape (H, W), grayscale image
    ph_trans: whether to apply photometric transforms
    """
    if ph_trans:
        # expand grayscale to RGB (HxWx1 → HxWx3)
        if img.ndim == 2:
            img = np.stack([img]*3, axis=-1)

        # convert to [0,1]
        img = torch.tensor(img).permute(2,0,1) / 255.0
        img = T.ToPILImage()(img)
        photometric_transforms = T.Compose([
            T.ColorJitter(
                brightness=0.5,
                contrast=0.5,
                saturation=0.5,
                hue=0.1
            ),
            T.RandomApply([T.GaussianBlur(3)], p=0.2),
        ])

        img = photometric_transforms(img)
        img = T.ToTensor()(img).permute(1,2,0).numpy()  # back to HWC numpy

        # random Gaussian noise
        if random.random() < 0.6:
            noise = np.random.normal(0, 0.05, img.shape)
            img = np.clip(img + noise, 0, 1)

        # # specular highlight
        # if random.random() < 0.5:
        #     img = add_specular_highlight(img, intensity=1.5, radius=random.randint(10,30))

        # # lens flare
        # if random.random() < 0.5:
        #     img = add_lens_flare(img, num_streaks=2)

        # tinting
        if random.random() < 0.5:
            tint_color = tuple(0.7 + 0.6*random.random() for _ in range(3))
            img = add_tint(img, tint_color)

        img = (255 * img).astype(np.uint8)
        # if np.random.uniform(0,1)>0.5:
        img = np.stack([cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)]*3,-1)
    return img

In [ ]:
def c2verts(centroids, h=None, szb=0.15, fix=False):
    N = centroids.shape[0]
    if fix:
        h = np.random.uniform(szb, szb, size=(N, 1))
    elif h is None:
        h = np.random.uniform(0.001, szb, size=(N, 1))  # (N, 1)
    base_offsets = np.array([
        [-1, -1, -1],
        [-1, -1,  1],
        [-1,  1, -1],
        [-1,  1,  1],
        [ 1, -1, -1],
        [ 1, -1,  1],
        [ 1,  1, -1],
        [ 1,  1,  1],
    ])  # (8, 3)
    offsets = base_offsets[None, :, :] * h[:, None, :]  # (N, 8, 3)
    vertices = centroids[:, None, :] + offsets  # (N, 8, 3)
    centroids_expanded = centroids[:, None, :]  # (N, 1, 3)
    all_points = np.concatenate([centroids_expanded, vertices], axis=1)  # (N, 9, 3)
    return all_points.reshape(-1, 3)#, oidx, iddx

In [ ]:
def get_camera_rotation(camera_xyz, target_xyz):
    camera_xyz = np.asarray(camera_xyz, dtype=np.float64)
    target_xyz = np.asarray(target_xyz, dtype=np.float64)
    direction = target_xyz - camera_xyz
    direction /= np.linalg.norm(direction)
    z_cam = -direction
    world_up = np.array([0.0, 0.0, 1.0])
    x_cam = np.cross(world_up, z_cam)
    if np.linalg.norm(x_cam) < 1e-8:
        world_up = np.array([0.0, 1.0, 0.0])
        x_cam = np.cross(world_up, z_cam)
    x_cam /= np.linalg.norm(x_cam)
    y_cam = np.cross(z_cam, x_cam)
    y_cam /= np.linalg.norm(y_cam)

    R_cam2world = np.column_stack([
        x_cam,
        y_cam,
        z_cam
    ])
    R_world2cam_blender = R_cam2world.T
    R_bb_c = np.array([
        [1,  0,  0],
        [0, -1,  0],
        [0,  0, -1]
    ], dtype=np.float64)
    R_world2cam = R_bb_c @ R_world2cam_blender

    R_b_g = R_world2cam.T @ R_c_b.T
    r_b_g = R.from_matrix(R_b_g).as_euler('xyz',degrees=True)

    return r_b_g

In [ ]:
## Canonical camera views ---- This can be updated as needed
K_mm_r = np.array([[385.464,0,330.348],[0,385.093,241.54],[0, 0, 1]])
sxc = 300/640 
syc = 300/480
kfr = np.array([[sxc,0,sxc],[0,syc,syc],[0,0,1]])
K_mm_rs = kfr * K_mm_r
def draw_shape(img, Km,tm,Rm,lnb,color=(255,0,0),thickness=1, rmr=True):
    imgn = np.array(img).astype(np.uint8)
    lnb_ = np.array(lnb)
    if rmr:
        if np.random.uniform(0,1)>0.5:
            Nbrs = int(np.random.uniform(0.65,1)*lnb.shape[0])
            vnd = random.sample(range(len(lnb)),Nbrs)
            lnb_ = lnb[vnd]

    for kk in range(lnb_.shape[0]):
        xyz = lnb_[kk]
        uvz = Km@Rm@(xyz.T - tm.reshape(-1,1))
        uv = uvz[:2,:] / uvz[2:,:]
        u,v = uv[0,:].astype(np.int16), uv[1,:].astype(np.int16)
        pt1, pt2 = (u[0],v[0]), (u[1],v[1])
        cv2.line(imgn, pt1, pt2, color, thickness)
    return imgn

In [ ]:
def proj_3d2d(xyz_ins, t1p, Rgc, wi=300, hi=300):
    uvs = K_mm_rs@Rgc@(xyz_ins.T - t1p.reshape(-1,1))
    uv = uvs[:2,:] / uvs[2:,:]
    uv = uv.T
    ff = uv[:,0]
    ff = (uv[:,0]>=0) * (uv[:,0]<wi) * (uv[:,1]>=0) * (uv[:,1]<hi) * (uvs[2,:]>0.1)
    return uv, ff

In [ ]:
def gen_cano_imgs(xyz_in_, lnb, rad=1, rmr=False):

    gimgs, aviz, UVs = [], [], []
    wi,hi= 300,300

    ptts = lnb.reshape(-1, 3)
    mnxx = np.min(ptts,0)
    mxxx = np.max(ptts,0)
    Lb, wb, hgt, sht = mxxx[1]-mnxx[1], mxxx[0]-mnxx[0], mxxx[-1], mnxx[-1]
    t1 = np.array([Lb,0.0,0.0])
    t3 = np.array([0.0,mnxx[1]-2*max([hgt,abs(sht)]),0])
    t5 = np.array([0.0,0.0,hgt+Lb])
    fc = 0.65
    t2 = np.array([fc*Lb,fc*Lb,-3*max([hgt,abs(sht)])])
    t4 = np.array([-fc*Lb,-fc*Lb,-3*max([hgt,abs(sht)])])
    t6 = np.array([-fc*Lb,fc*Lb,3*max([hgt,abs(sht)])])

    r1 = np.zeros(3)
    r3 = np.array([0,0,-90.0])
    r5 = np.array([0,-90,0])

    target_xyz = np.zeros(3) 
    r2 = get_camera_rotation(t2, target_xyz)
    r4 = get_camera_rotation(t4, target_xyz)
    r6 = get_camera_rotation(t6, target_xyz)

    Ts = [t1,t2,t3,t4,t5,t6] 
    Rs = [r1,r2,r3,r4,r5,r6]


    for pp in range(len(Ts)): #
        t1p = Ts[pp]
        r1p = Rs[pp]
        Rbg = np.array(R.from_euler('xyz',r1p,degrees=True).as_matrix())
        Rgc = np.array(R_c_b.T @ Rbg.T)
        uv_rnd, ff = proj_3d2d(xyz_in_, t1p, Rgc)
        UVs.append(uv_rnd)
        imgi = np.zeros((hi,wi,3)).astype(np.uint8)
        oimg_i = draw_shape(imgi, K_mm_rs,t1p,Rgc,lnb,thickness=rad, rmr=rmr)
        gimgs.append(oimg_i)
        aviz.append(ff.reshape(-1,1))

    vis_mat = np.hstack(aviz)
    imgs = np.stack(gimgs, axis=0)
    imgs = torch.from_numpy(imgs).float().permute(0,3,1,2)
    imgs = 2 * (imgs / 255.0) - 1.0
    return gimgs, imgs, UVs, vis_mat

In [ ]:
import os

In [ ]:
_DOMAIN_CACHE = {}
def prep_imgs_fast(adirs, idxs, xyz_vs, sz=300):
    imgs = []
    app_noise = np.random.uniform(0,1)>0.75 #False#False#False#
    for idc, jj in enumerate(idxs):
        img = cv2.imread(
            f"{adirs}img_{jj}.jpg")

        if app_noise:
            if np.random.uniform(0,1)>0.1:
                img = pt_trans(img, ph_trans=True)
        imgs.append(img)
    imgs = np.stack(imgs, axis=0)
    imgs = torch.from_numpy(imgs).float().permute(0,3,1,2) 
    imgs = 2 * (imgs / 255.0) - 1.0
    imgs = F.interpolate(
        imgs,
        size=(sz, sz),
        mode="bilinear",
        align_corners=False
    )
    return imgs

def load_domain(base, dom):
    """
    Loads and caches expensive npy files
    """
    if dom in _DOMAIN_CACHE:
        return _DOMAIN_CACHE[dom]
    data = {
        "pts": np.load(base + "pts.npy"),
        "sdf": np.load(base + "sdf.npy"),
        "lnb": np.load(base + "line_bars.npy")
    }
    _DOMAIN_CACHE[dom] = data
    return data

def proc_ptsnm(xyz_vs,sdf,Nmax):
    idi = sdf >= -0.001# mesh-conforming elements
    ido = sdf < -0.001# non-mesh conforming elements
    xyz_ina = xyz_vs[idi]
    xyz_outa = xyz_vs[ido]
    sdf_in = sdf[idi]
    sdf_ot = sdf[ido]
    nnmax1 = min(Nmax // 2, len(xyz_ina))
    nnmax2 = min(Nmax // 2, len(xyz_outa))
    rss1 = np.random.choice(len(xyz_ina), nnmax1, replace=False)
    rss2 = np.random.choice(len(xyz_outa), nnmax2, replace=False)
    xyz_in = xyz_ina[rss1]
    xyz_out = xyz_outa[rss2]
    cls_in = np.hstack([
        np.ones(len(xyz_in)),
        np.zeros(len(xyz_out))
    ])
    sc_in = np.hstack([
        sdf_in[rss1],
        sdf_ot[rss2]
    ])
    xyz_in = np.vstack([xyz_in, xyz_out])
    nmmc = min(Nmax, len(xyz_in) - 1)
    idm = random.sample(range(len(xyz_in)), nmmc)
    xyz_in = xyz_in[idm, :]
    cls_in = torch.tensor(cls_in[idm]).float()
    sc_in = torch.tensor(sc_in[idm]).float()
    xyz_in_ = c2verts(xyz_in, szb=0.05)
    return xyz_in_, cls_in, sc_in, xyz_in

def datagen4_mod_fast(dom, adirs_, sk=18, sz=300, Nmax=3000):
    data = load_domain(adirs_, dom)
    xyz_vs = data["pts"]
    sdf = data["sdf"]
    lnb = data["lnb"]

    xyz_in_, cls_in, sc_in, xyz_in = proc_ptsnm(xyz_vs,sdf,Nmax)
    n_imgs = len([f for f in os.listdir(adirs_) if f.startswith("img_") and f.endswith(".jpg")])
    inp_idx = random.sample(range(n_imgs), min(sk, n_imgs))
    all_inps_ = prep_imgs_fast(adirs_, inp_idx, xyz_in, sz=sz)
    gimgs, imgs, UVs, vis_mat = gen_cano_imgs(xyz_in_, lnb, rad=1, rmr=False)

    all_inps = [(imgs,all_inps_), UVs] 
    vis_mat_p = torch.tensor(vis_mat, dtype=torch.bool)
    vis_mat_p = vis_mat_p.view(len(cls_in), 9, -1).sum(dim=1) #uncomment later
    vis_mat_p = vis_mat_p > 0
    ff = vis_mat_p.sum(dim=1) >= 2

    return (
        gimgs,
        cls_in.long(),
        all_inps,
        vis_mat_p,
        ff.numpy(),
        sc_in)


## ---- training data (set in LM_paths.py; no need to edit) ----
## BridgeNet:            TRAIN_DIR             -> data/bridgenet/train
## Your synthetic data:  SYNTH_DIR / "train"   -> data/synthetic/train  (made by 01_LM_synthetic_data.py)
DATA_DIR = TRAIN_DIR
vals = list_domains(DATA_DIR)
assert len(vals) > 0, (f"No training domains found in {DATA_DIR}. "
                       "Download BridgeNet with:  python scripts/download_bridgenet.py --split train")
print(f"{len(vals)} training domains found in {DATA_DIR}")
def cross_data_fast(sk, sz, Nmax, ndoms=1):
    root_dir = str(DATA_DIR)
    doms = random.sample(vals, min(ndoms, len(vals)))
    aims, aclss, ainps, avizf, avls, ascs = [], [], [], [], [], []
    for dom in doms:
        bridge_id = f"{dom:05d}"
        adirs_ = os.path.join(root_dir,bridge_id,f"{bridge_id}_render/")
        out = datagen4_mod_fast(dom,adirs_,sk=sk,sz=sz,Nmax=Nmax)
        eimg, clss, inps, vizf, ff, scs = out
        aclss.append(clss)
        ainps.append(inps)
        avizf.append(vizf)
        aims.append(eimg) 
        avls.append(ff) 
        ascs.append(scs)
    return aims, aclss, ainps, avizf, avls, ascs

In [ ]:
def plotprogress(lss, cls_ac):
    clear_output(wait=True)
    plt.figure(figsize=(6 ,3))
    plt.plot(cls_ac, label='Occp.')
    plt.legend()
    plt.title('Cls. pred. accuracy')
    plt.figure(figsize=(8, 3))
    plt.plot(lss)
    plt.title('Total loss')
    plt.show()


In [ ]:
mod1 = group_occup_sdf().to(device)
mod1 = mod1.train()
optimz = optim.Adam([
    {'params': mod1.parameters(), 'lr': 1e-4}
])

In [ ]:
run_train = True
if run_train:
    nepoch = 50000
    losses = []
    cls_ac = []
    avg_cls = []
    prev_ac = 0.0
    for epoch in range(nepoch):
        out1 = cross_data_fast(sk=12, sz=224, Nmax=3000, ndoms=7)
        _, cls_in_, all_inps, viz_mat, ffs, sc_in_ = out1
        ff = np.hstack(ffs)
        cls_in = torch.hstack(cls_in_)[ff]
        sc_in = torch.hstack(sc_in_)[ff]

        print('cls_in: ', cls_in.shape)
        pred_xyz_ = mod1(all_inps, viz_mat)
        pred_xyz = pred_xyz_[ff,:] 
        losst = criterion3(pred_xyz[:,:3], cls_in.to(device))
        optimz.zero_grad()
        losst.backward()
        optimz.step()
        with torch.no_grad():
            losses.append(losst.item())
            pred_cls = torch.argmax(torch.softmax(pred_xyz[:,:3],1),1).cpu().flatten()
            print("pred_cls: ", pred_cls[:5])
            print("gt_cls: ", cls_in[:5])
            acc = torch.sum(pred_cls == cls_in) / len(cls_in) 
            cls_ac.append(100.0*acc)
            avg_cls.append(np.mean(cls_ac[-10:]))
            if (epoch % 20) == 0:
                plotprogress(losses, avg_cls)
                print("pred_acc: ", 100.0*acc)
            if (epoch % 100) == 0:
                cur_acc = np.mean(cls_ac[-10:])
                if cur_acc > prev_ac:
                    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
                    torch.save(mod1.state_dict(), TRAINED_WEIGHTS)
                    print('saved:', TRAINED_WEIGHTS)
                    prev_ac = cur_acc 